# MELD Preprocessing

For each utterance, extract:
- **audio/{split}/** — per-utterance `.wav` (extracted from mp4, 16kHz mono)
- **video/{split}/** — per-utterance `.mp4` (video-only, audio stripped)
- **text/{split}/** — per-utterance `.txt` (utterance text from CSV)
- **labels.csv** — unified label manifest

MELD clips are already pre-split per utterance (single speaker, single shot). No speaker cropping needed.

Split video locations:
- train: `Dataset/MELD.Raw/train/train_splits/dia{D}_utt{U}.mp4`
- dev:   `Dataset/MELD.Raw/dev/dev_splits_complete/dia{D}_utt{U}.mp4`
- test:  `Dataset/MELD.Raw/test/output_repeated_splits_test/dia{D}_utt{U}.mp4`

Output: `Dataset/Processed/MELD/`

In [10]:
import os
import subprocess
import csv
import pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm

In [ ]:
BASE      = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
MELD_ROOT = BASE / "Dataset/MELD.Raw"
OUT_ROOT  = BASE / "Dataset/Processed/MELD"

SPLITS = ["train", "dev", "test"]

# Source folders per split
VIDEO_SRC = {
    "train": MELD_ROOT / "train"  / "train_splits",
    "dev":   MELD_ROOT / "dev"    / "dev_splits_complete",
    "test":  MELD_ROOT / "test"   / "output_repeated_splits_test",
}

# Metadata CSVs
CSV_SRC = {
    "train": MELD_ROOT / "train" / "train_sent_emo.csv",
    "dev":   MELD_ROOT / "dev_sent_emo.csv",
    "test":  MELD_ROOT / "test_sent_emo.csv",
}

# Output dirs
for split in SPLITS:
    for modality in ["audio", "video", "text"]:
        (OUT_ROOT / modality / split).mkdir(parents=True, exist_ok=True)

## 1. Video Path Resolution

MELD test split has a dual naming convention (some files use the `final_videos_test` prefix).

In [12]:
def resolve_video_path(split: str, dia_id: int, utt_id: int) -> Path | None:
    """Return Path to the source mp4, or None if not found."""
    src_dir  = VIDEO_SRC[split]
    standard = src_dir / f"dia{dia_id}_utt{utt_id}.mp4"
    if standard.exists():
        return standard
    # Test split alternate naming convention
    prefixed = src_dir / f"final_videos_testdia{dia_id}_utt{utt_id}.mp4"
    if prefixed.exists():
        return prefixed
    return None

# Quick test on train split
p = resolve_video_path("train", 0, 0)
print(f"dia0_utt0 train path: {p}")

dia0_utt0 train path: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/MELD.Raw/train/train_splits/dia0_utt0.mp4


## 2. Playability Check

Screens out files with missing moov atom or zero fps (like `dia125_utt3.mp4`).

In [13]:
def is_playable(path: Path) -> bool:
    if not path.exists():
        return False
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=r_frame_rate",
         "-of", "default=noprint_wrappers=1:nokey=1",
         str(path)],
        capture_output=True, text=True
    )
    if result.returncode != 0 or not result.stdout.strip():
        return False
    fps_str = result.stdout.strip()  # e.g. '25/1'
    try:
        num, den = fps_str.split('/')
        return int(num) > 0
    except Exception:
        return False

## 3. Load All Records

In [14]:
all_records = []

for split in SPLITS:
    df = pd.read_csv(CSV_SRC[split])
    for _, row in df.iterrows():
        dia_id  = int(row["Dialogue_ID"])
        utt_id  = int(row["Utterance_ID"])
        src_vid = resolve_video_path(split, dia_id, utt_id)
        clip_name = f"dia{dia_id}_utt{utt_id}"
        all_records.append({
            "clip_name":   clip_name,
            "split":       split,
            "dia_id":      dia_id,
            "utt_id":      utt_id,
            "utterance":   str(row["Utterance"]),
            "speaker":     str(row["Speaker"]),
            "emotion":     str(row["Emotion"]),
            "sentiment":   str(row["Sentiment"]),
            "season":      int(row["Season"]),
            "episode":     int(row["Episode"]),
            "start_time":  str(row["StartTime"]),
            "end_time":    str(row["EndTime"]),
            "src_video":   str(src_vid) if src_vid else None,
        })

print(f"Total records: {len(all_records)}")
for split in SPLITS:
    n = sum(1 for r in all_records if r["split"] == split)
    missing = sum(1 for r in all_records if r["split"] == split and r["src_video"] is None)
    print(f"  {split}: {n} utterances | missing video: {missing}")

Total records: 13708
  train: 9989 utterances | missing video: 0
  dev: 1109 utterances | missing video: 1
  test: 2610 utterances | missing video: 0


## 4. Process Each Utterance (Audio + Video + Text)

Parallelised with `joblib`. For each utterance:
- **Audio**: FFmpeg `-vn -acodec pcm_s16le -ar 16000 -ac 1` → 16kHz mono WAV
- **Video**: FFmpeg `-an -c:v copy` → video stream only
- **Text**: write utterance string to `.txt`

In [15]:
def process_utterance(rec: dict) -> tuple[str, str]:
    """Returns (clip_name, status_string)."""
    clip  = rec["clip_name"]
    split = rec["split"]
    src   = rec["src_video"]

    dst_audio = OUT_ROOT / "audio" / split / f"{clip}.wav"
    dst_video = OUT_ROOT / "video" / split / f"{clip}.mp4"
    dst_text  = OUT_ROOT / "text"  / split / f"{clip}.txt"

    # Text is always writable regardless of video availability
    if not dst_text.exists():
        dst_text.write_text(rec["utterance"], encoding="utf-8")

    if src is None:
        return clip, "SKIP: video not found"

    src_path = Path(src)
    if not is_playable(src_path):
        return clip, "SKIP: unplayable"

    errors = []

    # Audio
    if not dst_audio.exists():
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(src_path),
            "-vn",
            "-acodec", "pcm_s16le",
            "-ar", "16000",
            "-ac", "1",
            str(dst_audio)
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            errors.append(f"audio: {r.stderr.strip()[:100]}")

    # Video
    if not dst_video.exists():
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(src_path),
            "-an",
            "-c:v", "copy",
            str(dst_video)
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            # Fallback: re-encode if stream copy fails
            cmd[-3:] = ["-c:v", "libx264", "-preset", "fast", "-crf", "18", str(dst_video)]
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0:
                errors.append(f"video: {r.stderr.strip()[:100]}")

    if errors:
        return clip, "error: " + " | ".join(errors)
    return clip, "ok"

results = Parallel(n_jobs=-1)(
    delayed(process_utterance)(rec)
    for rec in tqdm(all_records, desc="Processing MELD")
)

ok    = sum(1 for _, s in results if s == "ok")
skips = [(c, s) for c, s in results if s.startswith("SKIP")]
errs  = [(c, s) for c, s in results if s.startswith("error")]
print(f"OK: {ok} | Skipped: {len(skips)} | Errors: {len(errs)}")
if skips:
    for c, s in skips:
        print(f"  SKIP {c}: {s}")
if errs:
    for c, e in errs[:5]:
        print(f"  ERROR {c}: {e}")

Processing MELD: 100%|██████████| 13708/13708 [02:26<00:00, 93.38it/s] 


OK: 13706 | Skipped: 2 | Errors: 0
  SKIP dia125_utt3: SKIP: unplayable
  SKIP dia110_utt7: SKIP: video not found


## 5. Write labels.csv

In [16]:
# Build status lookup
status_map = {clip: status for clip, status in results}

labels_path = OUT_ROOT / "labels.csv"
fieldnames = [
    "clip_name", "split", "dia_id", "utt_id",
    "utterance", "speaker", "emotion", "sentiment",
    "season", "episode", "start_time", "end_time",
    "audio_path", "video_path", "text_path", "status"
]

with open(labels_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for rec in all_records:
        clip  = rec["clip_name"]
        split = rec["split"]
        status = status_map.get(clip, "unknown")
        writer.writerow({
            "clip_name":  clip,
            "split":      split,
            "dia_id":     rec["dia_id"],
            "utt_id":     rec["utt_id"],
            "utterance":  rec["utterance"],
            "speaker":    rec["speaker"],
            "emotion":    rec["emotion"],
            "sentiment":  rec["sentiment"],
            "season":     rec["season"],
            "episode":    rec["episode"],
            "start_time": rec["start_time"],
            "end_time":   rec["end_time"],
            "audio_path": f"audio/{split}/{clip}.wav",
            "video_path": f"video/{split}/{clip}.mp4",
            "text_path":  f"text/{split}/{clip}.txt",
            "status":     status,
        })

print(f"labels.csv written: {labels_path}")
print(f"Total rows: {len(all_records)}")

labels.csv written: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD/labels.csv
Total rows: 13708


## 6. Verification

In [17]:
df = pd.read_csv(labels_path)
print(f"Labels CSV rows: {len(df)}")

print(f"\nPer-split counts:")
print(df["split"].value_counts())

print(f"\nEmotion distribution:")
print(df["emotion"].value_counts())

print(f"\nStatus breakdown:")
print(df["status"].value_counts())

for split in SPLITS:
    audio_n = len(list((OUT_ROOT / "audio" / split).glob("*.wav")))
    video_n = len(list((OUT_ROOT / "video" / split).glob("*.mp4")))
    text_n  = len(list((OUT_ROOT / "text"  / split).glob("*.txt")))
    print(f"\n{split}: audio={audio_n}, video={video_n}, text={text_n}")

# Spot check
ok_rows = df[df["status"] == "ok"]
sample = ok_rows.iloc[0]
print(f"\nSpot check — {sample.clip_name} ({sample.split}):")
print(f"  emotion: {sample.emotion}")
print(f"  text:    {sample.utterance[:80]}")
audio_path = OUT_ROOT / sample.audio_path
video_path = OUT_ROOT / sample.video_path
print(f"  audio exists: {audio_path.exists()}")
print(f"  video exists: {video_path.exists()}")

# Expected counts (MELD standard splits; -1 for dia125_utt3 SKIP)
expected = {"train": 9989, "dev": 1108, "test": 2610}
print(f"\nCount verification:")
for split in SPLITS:
    n = len(df[df["split"] == split])
    exp = expected[split]
    ok_flag = "✓" if abs(n - exp) <= 1 else "✗"
    print(f"  {split}: {n} / {exp} expected  {ok_flag}")

Labels CSV rows: 13708

Per-split counts:
split
train    9989
test     2610
dev      1109
Name: count, dtype: int64

Emotion distribution:
emotion
neutral     6436
joy         2308
surprise    1636
anger       1607
sadness     1002
disgust      361
fear         358
Name: count, dtype: int64

Status breakdown:
status
ok    13708
Name: count, dtype: int64

train: audio=9988, video=9988, text=9989

dev: audio=1108, video=1108, text=1109

test: audio=2610, video=2610, text=2610

Spot check — dia0_utt0 (train):
  emotion: neutral
  text:    also I was the point person on my companys transition from the KL-5 to GR-6 sys
  audio exists: True
  video exists: True

Count verification:
  train: 9989 / 9989 expected  ✓
  dev: 1109 / 1108 expected  ✓
  test: 2610 / 2610 expected  ✓
